# EMD-Transformer-BiLSTM — Prediksi AQI Jakarta
Dataset: Jakarta US_AQI (Open-Meteo CAMS + weather)
Pipeline: EMD(12 IMF) → per-IMF Transformer-BiLSTM → Linear fallback → fuse via BiLSTM


In [ ]:
import os

REPO_URL = "https://github.com/madajabbar/riset-aqi-jakarta.git"
BRANCH = "main"

if not os.path.exists("riset-aqi-jakarta"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    %cd riset-aqi-jakarta
    !git pull origin {BRANCH}
    %cd ..

%cd riset-aqi-jakarta

In [ ]:
!pip install -q torch scikit-learn matplotlib pandas numpy PyEMD

In [ ]:
# Verify Jakarta dataset
import os
aqi = "our_data/jakarta_aqi.csv"
met = "our_data/jakarta_met.csv"
assert os.path.exists(aqi), f"{aqi} missing!"
assert os.path.exists(met), f"{met} missing!"
print(f"AQI: {os.path.getsize(aqi):,} bytes | Weather: {os.path.getsize(met):,} bytes")

In [ ]:
# Train weather model (EMD-Transformer-BiLSTM + CAMS weather)
!PYTHONUNBUFFERED=1 python our_model/train.py --weather --start 2022-08-15 --end 2026-10-01 2>&1 | tee our_model/results/full_weather.log

In [ ]:
# Train base model (ABLASI: tanpa weather)
!PYTHONUNBUFFERED=1 python our_model/train.py --start 2022-08-15 --end 2026-10-01 2>&1 | tee our_model/results/full_base.log

In [ ]:
# Display results
from IPython.display import Image, display
import glob

for f in sorted(glob.glob("our_model/results/*.png")):
    print(f"--- {f} ---")
    display(Image(filename=f))

import pandas as pd
for f in sorted(glob.glob("our_model/results/*.csv")):
    print(f"\n{f}:")
    print(pd.read_csv(f, header=None))